# 04. e + Ba⁺: мишень, A, сечения 6s → 6p, 5d и из метастабилей 5d

Линии: **455.403** (6p ²P₃/₂ → 6s), **493.408** (6p ²P₁/₂ → 6s); ветви 614.171, 585.367, 649.690
(→ 5d) — для калибровки чувствительности.

Модель `ba2`: один валентный электрон над [Xe]: 6s, 5d, 7s, 6d, 8s, 7d | 6p, 7p, 4f, 5f
(17 состояний). Поляризации остова нет — проверяем по энергиям и A.

**Валидация**:
1. энергии против NIST (пороги всё равно экспериментальные);
2. A(455.4) = 1.11·10⁸, A(493.4) = 9.53·10⁷, A(614.2) = 4.12·10⁷, A(585.4) = 6.0·10⁶,
   A(649.7) = 3.10·10⁷ с⁻¹ (NIST, класс B), ветвления 6p;
3. времена жизни 5d₃/₂, 5d₅/₂ (E2, десятки секунд) против измеренных;
4. связанные состояния (N+1) = уровни Ba I против NIST;
5. сечение 6s → 6p: асимптотика Бете с f из расчёта; сравнение с опубликованными данными
   (впишите их в `LIT`).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # эта папка: nbtools.py, run_model.py, models.py
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
SPECTRUM, SHEET = DATA / "A25_second.xlsx", "data2"   # измеренный спектр
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST (nist_*.csv) и список линий
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import crm, reference
import nbtools as nb
%matplotlib inline

In [ ]:
JMAX = 20           # Ba+: дёшево, можно сразу 20-30 (дипольные 6s->6p сходятся медленно)
job = nb.Job(RUNS / "ba2", "ba2", stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=30, de=0.005, multipoles="E1,E2")
job.start()

In [ ]:
job.status()

## 1. Энергии

In [ ]:
W = RUNS / "ba2"
print((W / "target_table.txt").read_text())

## 2–3. A, ветвления, времена жизни

In [ ]:
NIST_A = {455.403: 1.11e8, 493.408: 9.53e7, 614.171: 4.12e7, 585.367: 6.00e6, 649.690: 3.10e7,
          452.493: 6.63e7, 489.993: 1.04e8, 413.065: 2.18e8, 389.178: 2.17e8}
rows = nb.read_csv(W / "transitions_E1.csv")
for r in rows:
    wl = nb.fnum(r["wl_air_nm"])
    hit = [k for k in NIST_A if wl and abs(k - wl) < 0.05]
    if hit:
        a = float(r["A_exp"])
        print(f"{hit[0]:9.3f}  {r['upper_nist']:>18s} -> {r['lower_nist']:<18s}  DBSR {a:.3e}  NIST {NIST_A[hit[0]]:.3e}"
              f"  ratio {a / NIST_A[hit[0]]:.2f}  vel/len {r['gauge_ratio']}")
for kind in ("E1", "M1", "E2"):
    f = W / f"transitions_{kind}.csv"
    if f.exists():
        _, tot, _ = nb.branching_from_csv(f)
        print(kind, {u: f"{1 / t:.3g} с" for u, t in sorted(tot.items()) if u <= 5})

## 4. Связанные состояния e + Ba⁺ = уровни Ba I

In [ ]:
# отдельный короткий прогон (J <= 4): dbsr_hd3 itype=-1 на заново построенных матрицах
jb = nb.Job(RUNS / "ba2_bound", "ba2", stage="bound", jmax=4, cores=CORES, mem=MEM, hd_threads=HD_THREADS)
jb.start()

In [ ]:
jb.status()
lv1 = db.nist.fetch_levels("Ba I")
for b in sorted(nb.read_csv(RUNS / "ba2_bound" / "bound_states.csv"), key=lambda b: float(b["E_cm"]))[:20]:
    e = float(b["E_cm"])                      # над самым нижним связанным состоянием (Ba I 6s2)
    near = min(lv1, key=lambda l: abs(l.energy_cm - e))
    print(f"{b['label']:>14s}  E = {e:9.1f} см-1  (связь {float(b['E_bind']):8.4f})   "
          f"ближайший NIST: {near.config} {near.term} J={near.J:g}  {near.energy_cm:9.1f}  Δ = {e - near.energy_cm:+7.0f}")

## 5. Сечения и константы скорости

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz")
tg = db.Target.load(W / "target")
st = {s.name: s for s in tg.states}
lab = {n: f"{st[n].nist_label}" for n in cs.names}
gs = cs.names[0]
LIT = {}            # литературные σ(6s->6p): {"источник": (E_эВ, σ_см2)}
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for j in cs.names[1:5]:
    axs[0].plot(cs.incident_energy(gs), cs.sigma(gs, j) / 1e-16, lw=0.8, label=f"6s → {lab[j]}")
for k, (E, s) in LIT.items():
    axs[0].plot(E, s / 1e-16, "o", ms=3, label=k)
for i in cs.names[1:3]:
    for j in cs.names[3:5]:
        axs[1].plot(cs.incident_energy(i), cs.sigma(i, j) / 1e-16, lw=0.8, label=f"{lab[i]} → {lab[j]}")
for ax in axs:
    ax.set_xscale("log"); ax.set_xlabel("E, эВ"); ax.set_ylabel("σ, 10⁻¹⁶ см²"); ax.legend(fontsize=7)
plt.show()
TE = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 5.0])
for i in cs.names[:3]:
    for j in cs.names[1:5]:
        if st[j].exp_energy_cm > st[i].exp_energy_cm:
            print(f"{lab[i]:>12s} -> {lab[j]:<12s}", " ".join(f"{x:.2e}" for x in cs.rate(i, j, TE * 11604.5)))

## CRM Ba II: отношение 455.4 / 493.4 против ne

²P₁/₂ из 5d₅/₂ дипольно не возбуждается, ²P₃/₂ — возбуждается: отношение растёт с
заселённостью 5d₅/₂, то есть с ne·τ. τ — время пролёта атома/иона бария через зону
наблюдения (Ba при ~1500 K: v ≈ 430 м/с; ионы быстрее).

In [ ]:
data = crm.CRMData.load(W / "crm_ba2.json")
n32 = [n for n in data.names if "2P" in (data.labels[data.index[n]] or "") and data.g[data.index[n]] == 4][0]
n12 = [n for n in data.names if "2P" in (data.labels[data.index[n]] or "") and data.g[data.index[n]] == 2][0]
g = data.names[0]
NE = np.logspace(9, 13, 30)
for tau in (2e-6, 2e-5):
    for te in (0.8, 1.1, 2.0):
        y = [crm.line_emission(data, p := crm.steady_state(data, ne, te, tau=tau), n32, g) /
             crm.line_emission(data, p, n12, g) for ne in NE]
        plt.semilogx(NE, y, label=f"τ = {tau:g} с, Te = {te} эВ")
plt.xlabel("ne, см⁻³"); plt.ylabel("I(455.4) / I(493.4), фотоны"); plt.legend(fontsize=7); plt.show()